In [16]:
import requests
import zipfile
import io
import os
import pandas as pd
import nltk, re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib
import numpy as np
import re

'''
# Get the data and unpack it
# URL of the dataset
url = "https://academy.hackthebox.com/storage/modules/292/skills_assessment_data.zip"
# Download the dataset
response = requests.get(url)
if response.status_code == 200:
    print("Download successful")
else:
    print("Failed to download the dataset")

# Extract the zip   
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall("skills_assessment_data")
    print("Extraction successful")

# List the extracted files
extracted_files = os.listdir("skills_assessment_data")
print("Extracted files:", extracted_files)

'''
# Set file path
train_path = r"skills_assessment_data/train.json"

#Import json to df
df=pd.read_json(train_path)

# Display basic information about the dataset
print("-------------------- HEAD --------------------")
print(df.head())
print("-------------------- DESCRIBE --------------------")
print(df.describe())
print("-------------------- INFO --------------------")
print(df.info())

# Remove duplicates if any
df = df.drop_duplicates()

# Check for missing values
print("Missing values:\n", df.isnull().sum())

# Check for duplicates
print("Duplicate entries:", df.duplicated().sum())

print("=== BEFORE ANY PREPROCESSING ===") 
print(df.head(5))

# Convert all message text to lowercase
df["text"] = df["text"].str.lower()
print("\n=== AFTER LOWERCASING ===")
print(df["text"].head(5))

# Remove non-essential punctuation and numbers, keep useful symbols like $ and !
df["text"] = df["text"].apply(lambda x: re.sub(r"[^a-z\s$!]", "", x))
print("\n=== AFTER REMOVING PUNCTUATION & NUMBERS (except $ and !) ===")
print(df["text"].head(5))

# Split each message into individual tokens
df["text"] = df["text"].apply(word_tokenize)
print("\n=== AFTER TOKENIZATION ===")
print(df["text"].head(5))

# Define a set of English stop words and remove them from the tokens
stop_words = set(stopwords.words("english"))
df["text"] = df["text"].apply(lambda x: [word for word in x if word not in stop_words])
print("\n=== AFTER REMOVING STOP WORDS ===")
print(df["text"].head(5))

# Stem each token to reduce words to their base form
stemmer = PorterStemmer()
df["text"] = df["text"].apply(lambda x: [stemmer.stem(word) for word in x])
print("\n=== AFTER STEMMING ===")
print(df["text"].head(5))

df["text"] = df["text"].apply(lambda x: " ".join(x))
print("\n=== AFTER JOINING TOKENS BACK INTO STRINGS ===")
print(df["text"].head(5))

# Initialize CountVectorizer with bigrams, min_df, and max_df to focus on relevant terms
# vectorizer = CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))

# Fit and transform the message column
# X = vectorizer.fit_transform(df["text"])

# Labels (target variable)
y = df["label"]

# Build the pipeline by combining vectorization and classification
pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
    # ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])

# Define stratified_cv
stratified_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Define the parameter grid for hyperparameter tuning
'''
param_grid = {
    "classifier__alpha": [0.1, 1.0, 10.0, 100.0]
}
'''

param_grid = {
    "classifier__C": [0.1, 1.0, 10.0, 100.0],  # 4 values instead of 12
    "classifier__class_weight": ['balanced'],  # 2 values
    "vectorizer__max_features": [20000],  # 2 values instead of 5
    "vectorizer__ngram_range": [(1, 3)],  # 2 values instead of 3
}

# Perform the grid search with 5-fold cross-validation and the F1-score as metric
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=stratified_cv,
    scoring="accuracy"
)

# Fit the grid search on the full dataset
grid_search.fit(df["text"], y)

# Extract the best model identified by the grid search
best_model = grid_search.best_estimator_
print("Best model parameters:", grid_search.best_params_)

# Save the trained model to a file for future use
model_filename = 'skills_assessment.joblib'
joblib.dump(best_model, model_filename)

print(f"Model saved to {model_filename}")

-------------------- HEAD --------------------
                                                text  label
0  Bromwell High is a cartoon comedy. It ran at t...      1
1  Homelessness (or Houselessness as George Carli...      1
2  Brilliant over-acting by Lesley Ann Warren. Be...      1
3  This is easily the most underrated film inn th...      1
4  This is not the typical Mel Brooks film. It wa...      1
-------------------- DESCRIBE --------------------
             label
count  25000.00000
mean       0.50000
std        0.50001
min        0.00000
25%        0.00000
50%        0.50000
75%        1.00000
max        1.00000
-------------------- INFO --------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    25000 non-null  object
 1   label   25000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 390.8+ KB
None
Missing values:

In [ ]:
# Validation

# Set file path
test_path = r"skills_assessment_data/test.json"

#Import json to df
test_data=pd.read_json(test_path)

# Strip the text column out to a list
if 'text' in test_data.columns:
    test_reviews = test_data['text'].tolist()
    
# Preprocess function that mirrors the training-time preprocessing
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s$!]", "", text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)

# Preprocess and vectorize text
processed_text = [preprocess_text(text) for text in test_reviews]

# Transform preprocessed messages into feature vectors
X_new = best_model.named_steps["vectorizer"].transform(processed_text)

# Predict with the trained classifier
predictions = best_model.named_steps["classifier"].predict(X_new)
prediction_probabilities = best_model.named_steps["classifier"].predict_proba(X_new)

# Display predictions and probabilities for each evaluated message
for i, og_text in enumerate(test_reviews):
    prediction = "positive" if predictions[i] == 1 else "negative"
    positive_probability = prediction_probabilities[i][1]  # Probability of being positive
    negative_probability = prediction_probabilities[i][0]   # Probability of being negative
    
    print(f"Message: {og_text}")
    print(f"Prediction: {prediction}")
    print(f"Positive Probability: {positive_probability:.2f}")
    print(f"Negative Probability: {negative_probability:.2f}")
    print("-" * 50)